# Catch a correct proof of the wrong theorem

*Adversarial Mathematics From First Principles*, [chapter 5](https://programmer.ie/books/math/05-chapter/) · [worked example](https://programmer.ie/examples/math/05-chapter/)

Run the cells in order. Everything below the program is taken from the laboratory's recorded, hash-verified run; the program is a small, self-contained demonstration of the same mathematics.

A proof assistant checks that a declaration proves *its own* statement. If a
tool then credits that declaration with a claim by its name alone, a proof of
something else can be reported as a proof of the claim. This file declares
`zero_add_left` four times: the claim, the claim proved differently, `True`,
and a weakened version. Lean accepts all four, correctly.

## Run it

Colab has no Lean, so the first cell downloads the official Lean 4.35.0 release (core only, about 600 MB, a minute or two). The second writes the file and the third checks it. The two type mismatches are the expected result.

In [1]:
!apt-get -qq install -y zstd > /dev/null
!test -x lean-4.35.0-linux/bin/lean || curl -sSfL https://github.com/leanprover/lean4/releases/download/v4.35.0/lean-4.35.0-linux.tar.zst | tar --zstd -x
!lean-4.35.0-linux/bin/lean --version

Lean (version 4.35.0, x86_64-unknown-linux-gnu, commit eb6cafb7290e8bd5f2937522d8d804f58f02bafc, Release)


In [2]:
%%writefile wrong_theorem.lean
-- The wrong theorem, in core Lean 4 (no Mathlib). Paste into a Lean 4 editor.
-- Four declarations share one name. Lean accepts all four, correctly:
-- each proves its own statement. Only two of them prove the requested claim.

namespace Honest      -- the claim, by induction
theorem zero_add_left : ∀ n : Nat, 0 + n = n := by
  intro n
  induction n with
  | zero => rfl
  | succ n ih => rw [Nat.add_succ, ih]
end Honest

namespace Revised     -- the same claim: renamed variable, different proof
theorem zero_add_left : ∀ m : Nat, 0 + m = m := fun m => Nat.zero_add m
end Revised

namespace Substituted -- same name, a different statement
theorem zero_add_left : True := trivial
end Substituted

namespace Weakened    -- same name, only up to 10
theorem zero_add_left : ∀ n : Nat, n ≤ 10 → 0 + n = n := by
  intro n _
  exact Nat.zero_add n
end Weakened

-- A name-only gate sees four declarations called `zero_add_left` and credits all four.
-- The statement gate asks Lean to accept each one AS A PROOF OF THE PINNED CLAIM:
example : ∀ n : Nat, 0 + n = n := Honest.zero_add_left       -- accepted
example : ∀ n : Nat, 0 + n = n := Revised.zero_add_left      -- accepted
example : ∀ n : Nat, 0 + n = n := Substituted.zero_add_left  -- refused
example : ∀ n : Nat, 0 + n = n := Weakened.zero_add_left     -- refused

#print axioms Honest.zero_add_left

Writing wrong_theorem.lean


In [3]:
!lean-4.35.0-linux/bin/lean wrong_theorem.lean

wrong_theorem.lean:31:34: error: Type mismatch
  Substituted.zero_add_left
has type
  True
but is expected to have type
  ∀ (n : Nat), 0 + n = n
wrong_theorem.lean:32:34: error: Type mismatch
  Weakened.zero_add_left
has type
  ∀ (n : Nat), n ≤ 10 → 0 + n = n
but is expected to have type
  ∀ (n : Nat), 0 + n = n
'Honest.zero_add_left' does not depend on any axioms


## What the output shows

The two type mismatches are the point. Asked to accept each declaration *as a
proof of the pinned statement*, Lean refuses the substituted and weakened ones,
while the honest and revised proofs pass; a gate that looked only at the name
would have credited all four. This was a real defect in this project's own
earlier tooling, found by adversarial review; Lean was never wrong.

## What the laboratory checked

The recorded run `20261113T232811Z-24153376` ([evidence](https://github.com/ernanhughes/math/tree/main/experiments/investigations/p04-wrong-theorem/runner-runs/2026-11-13-pinned)) is the book's result; the program above is a reader-sized demonstration of the same mathematics. **Evidence:** Fresh kernel check in core Lean: statement-attachment tests on four cases. **Status:** `kernel_checked_correspondence_pending`.

> A kernel can correctly accept a declaration whose statement is not the claim it is credited with, and a gate that attaches evidence by declaration name then reports the wrong claim verified; binding the declaration to an independently pinned statement, checked by the kernel, refuses the substitution and the weakening while accepting a genuine proof with a renamed variable or a different proof.

| Check | System | Recorded outcome |
| --- | --- | --- |
| `inputs` | Pinned inputs | all inputs match their pins |
| `demo` | Python driving core Lean (both attachment gates on four cases) | exit 0, verdict attribution_defect_shown_and_repaired |

Failure controls: **1 of 1** rejected or refused as required (`tampered-claim`).
Correspondence `P04-CORR-01` (does the formal statement say what the claim means?): **pending the author's approval**.
Contract `P04-WRONG-THEOREM` version 2, sha256 `cda883e7b96e99e5…`; `result.json` sha256 `7ac90a591a982521…`.

## Not shown

As the laboratory's own record states:

- That the statement gate refuses every possible attack: only the registered cases are tested.
- That the pinned statement says what the English claim means: that is the correspondence, which only a person can approve.
- Anything about MATH-INV-02's own 84 declarations, whose repaired identity check is recorded separately (see the Haraux investigation).

## Reproduce or challenge it

To reproduce the laboratory's run, build the `math-env` image from a clone of the repository ([Getting Started](/books/math/29-chapter/)) and run:

```bash
img=$(docker image inspect math-env --format '{{.Id}}')
docker run --rm --network none -e UV_OFFLINE=1 -e MATHENV_IMAGE_ID=$img -v "${PWD}:/source:ro" -v "${PWD}/out:/out" math-env uv run --offline math-engine proof wrong-theorem --out /out/proof-wrong-theorem
```

This Docker form was tested for `fermat`, `kempe` and `simpson`, not separately for `wrong-theorem`. To re-verify the recorded run itself without Docker, `math-engine proof --verify experiments/investigations/p04-wrong-theorem/runner-runs/2026-11-13-pinned` (R-V04).

Examine the statement gate. Construct a declaration that both the name-only
gate and the statement gate would accept but that still does not establish the
intended claim (consider definitional tricks, extra axioms and changed
notation). Say whether the four registered cases cover the attacks that matter,
and propose additional cases.